# **Node Prominence by Metric (NPM)**

In [1]:
import os
import sys
# from itertools import combinations

import pandas as pd

# Add project root to Python's path
sys.path.append(os.path.abspath(os.path.join('..')))

# Remove the groups that are not of interest
GROUPS = ['Basal-like', 'Luminal A']

# Metrics calculated in the individual analysis
METRICS = {
    'bipartite': [
        'betweenness_centrality', 
        'closeness_centrality', 
        'degree_centrality', 
        'redundancy_coefficient',
    ],
    'tripartite': [
        'pathway_reach',
        'functional_impact',
    ],
}

# Thresholds for node classification
LOWER_NODE_RANKING_THRESHOLD = 0.25
UPPER_NODE_RANKING_THRESHOLD = 0.50
# JACCARD_NODE_RANKING_THRESHOLD = 0.25

# Functions

## NPM Calculation

### Rank Nodes by Metric Values

In [2]:
def rank_nodes_by_metric_values(
    nodes_df,
    partition,
    metric,
    ascending=False,
):
    ranked_nodes = dict()

    groups = nodes_df['group'].unique()

    for group in groups:
        ranked_df = (
            nodes_df
            .query('(group == @group) and (type == @partition)')
            .dropna(subset=[metric])
            .sort_values(
                by=[metric, 'id'],
                ascending=[ascending, True],
            )
        )

        ranked_nodes[group] = ranked_df['id'].tolist()

    return ranked_nodes

### Identify Specific Nodes

In [3]:
def identify_specific_nodes(ranked_nodes):
    specific = dict()
    shared = dict()
    processed_nodes = set()

    groups = ranked_nodes.keys()

    for group1 in groups:
        group1_nodes = ranked_nodes[group1]

        lower_threshold = int(
            LOWER_NODE_RANKING_THRESHOLD * len(group1_nodes)
        )

        for node in group1_nodes[:lower_threshold]:
            if node in processed_nodes:
                continue

            node_groups = [group1]

            for group2 in groups:
                if group2 == group1:
                    continue

                group2_nodes = ranked_nodes[group2]

                upper_threshold = int(
                    UPPER_NODE_RANKING_THRESHOLD * len(group2_nodes)
                )

                if node in group2_nodes[:upper_threshold]:
                    node_groups.append(group2)

            processed_nodes.add(node)
            node_groups = sorted(node_groups)

            if len(node_groups) == 1:
                specific[node] = node_groups
            else:
                shared[node] = node_groups

    return {
        'specific': specific,
        'shared': shared,
    }

### Compute Shared Node Similarity

In [4]:
def compute_shared_similarity(edges_df, node_specificity):
    shared = node_specificity['shared']
    shared_similarity = dict()

    for node, groups in shared.items():
        g1, g2 = groups

        g1_neighbors = set(
            edges_df
            .query('(source == @node) and (group == @g1)')
            ['target']
            .tolist()
        )

        g2_neighbors = set(
            edges_df
            .query('(source == @node) and (group == @g2)')
            ['target']
            .tolist()
        )

        intersection = g1_neighbors & g2_neighbors
        union = g1_neighbors | g2_neighbors

        similarity = (
            len(intersection) / len(union)
            if union
            else 0.0
        )

        shared_similarity[node] = [
            g1,
            g2,
            round(similarity, 3),
        ]

    node_specificity['shared_similarity'] = shared_similarity

    return node_specificity

### Compute Node Prominence by Metric

In [5]:
def compute_node_prominence(
    concat_edges_df,
    concat_nodes_df,
    partitions=('MicroRNA',),
):
    node_prominence = dict()

    # Keep only microRNA-mRNA edges for neighborhood similarity
    edges_df = (
        concat_edges_df
        .dropna(subset=['correlation'])
        .drop_duplicates(ignore_index=True)
    )

    # Keep only nodes from the largest bipartite CC for comparison
    comparison_df = (
        concat_nodes_df
        .loc[concat_nodes_df['is_in_largest_component'].eq(1)]
        .copy()
    )

    metrics = (
        METRICS['bipartite']
        + METRICS['tripartite']
    )

    for partition in partitions:
        node_prominence[partition] = dict()

        for metric in metrics:
            ranking_orders = (
                [False, True]
                if metric == 'redundancy_coefficient'
                else [False]
            )

            node_prominence[partition][metric] = dict()

            for ascending in ranking_orders:
                ranked_nodes = rank_nodes_by_metric_values(
                    comparison_df,
                    partition,
                    metric,
                    ascending,
                )

                classifications = identify_specific_nodes(
                    ranked_nodes
                )

                if classifications['shared']:
                    classifications = compute_shared_similarity(
                        edges_df,
                        classifications,
                    )

                ranking = (
                    'ascending'
                    if ascending
                    else 'descending'
                )

                node_prominence[partition][metric][
                    ranking
                ] = classifications

    return node_prominence

### Print Node Prominence By Metric

In [6]:
def print_node_prominence(node_prominence):
    for partition, metrics in node_prominence.items():
        print(f'===== Tripartite Networks - {partition} Nodes =====')

        for metric, rankings in metrics.items():
            subtitle = metric.replace('_', ' ').title()

            for ranking, classifications in rankings.items():
                print(
                    f'\n##### {subtitle} - '
                    f'{ranking.title()} Ranking #####'
                )

                for classification, elements in classifications.items():
                    print(f'\n> {classification} = {len(elements)}')

                    for node, groups in sorted(elements.items()):
                        print(f'- {node}: {groups}')

        print()

### Save the Results in Tables

In [7]:
def add_npm_results_to_nodes(nodes_df, node_prominence):
    result_df = nodes_df.copy()

    for partition, metrics in node_prominence.items():
        for metric, rankings in metrics.items():
            for ranking, classifications in rankings.items():
                column = f'{metric}_{ranking}'
                npm_values = {}

                # Add specific nodes
                specific_nodes = classifications.get('specific', {})

                for node, groups in specific_nodes.items():
                    for group in groups:
                        npm_values[(node, group)] = [['specific']]

                # Add shared nodes
                shared_nodes = classifications.get('shared', {})
                similarities = classifications.get(
                    'shared_similarity',
                    {},
                )

                for node, groups in shared_nodes.items():
                    g1, g2, similarity = similarities[node]

                    npm_values[(node, g1)] = [
                        ['shared', g2, similarity]
                    ]
                    npm_values[(node, g2)] = [
                        ['shared', g1, similarity]
                    ]

                # Add NPM results to node table
                result_df[column] = [
                    npm_values.get((node, group), [])
                    for node, group in zip(
                        result_df['id'],
                        result_df['group'],
                    )
                ]

    return result_df

In [8]:
def save_group_network_data(edges_df, nodes_df):
    groups = nodes_df['group'].unique()

    for group in groups:
        group_name = group.lower().replace(' ', '-')

        group_edges_df = (
            edges_df
            .query('group == @group')
            .drop(columns=['group'])
            .reset_index(drop=True)
        )

        group_nodes_df = (
            nodes_df
            .query('group == @group')
            .drop(columns=['group'])
            .reset_index(drop=True)
        )

        group_edges_df.to_csv(
            f'{group_name}_edges.csv',
            index=False,
        )

        group_nodes_df.to_csv(
            f'{group_name}_nodes.csv',
            index=False,
        )

# Load the Data

In [9]:
TRI_EDGES_FILE = 'tripartite-network-edges.csv'
TRI_NODES_FILE = 'tripartite-network-nodes.csv'
BI_NODES_FILE = 'interaction-network-nodes.csv'

edges_dfs = []
nodes_dfs = []

for group in GROUPS:
    group_dir = group.lower().replace(' ', '-')
    dir_path = os.path.join(
        '..',
        'data',
        'processed',
        'network-analysis',
        group_dir,
    )

    # Load tripartite network
    edges_df = pd.read_csv(
        os.path.join(dir_path, TRI_EDGES_FILE)
    )

    nodes_df = pd.read_csv(
        os.path.join(dir_path, TRI_NODES_FILE)
    )

    # Load bipartite node attributes
    bipartite_nodes_df = pd.read_csv(
        os.path.join(dir_path, BI_NODES_FILE)
    )

    # Select attributes that are not already present in the tripartite table
    bipartite_cols = [
        col
        for col in bipartite_nodes_df.columns
        if col == 'id' or col not in nodes_df.columns
    ]

    # Add all bipartite-specific node attributes
    nodes_df = pd.merge(
        left=nodes_df,
        right=bipartite_nodes_df[bipartite_cols],
        how='left',
        on='id',
    )

    # Put bipartite metrics before tripartite metrics
    metric_cols = (
        METRICS['bipartite']
        + METRICS['tripartite']
    )

    other_cols = [
        col for col in nodes_df.columns
        if col not in metric_cols
    ]

    nodes_df = nodes_df[
        other_cols + metric_cols
    ]

    edges_dfs.append(
        edges_df.assign(group=group)
    )

    nodes_dfs.append(
        nodes_df.assign(group=group)
    )

concat_edges_df = pd.concat(
    edges_dfs,
    ignore_index=True,
)

concat_nodes_df = pd.concat(
    nodes_dfs,
    ignore_index=True,
)

In [10]:
concat_edges_df

,source,target,mirtarbase,correlation,qvalue,group
0,hsa-miR-182-5p,RAB27A,NaN,-0.564,0.000,Basal-like
1,hsa-miR-18a-5p,ORAI3,MIRT050737,-0.545,0.000,Basal-like
2,hsa-miR-18a-5p,ZBTB4,NaN,-0.532,0.001,Basal-like
3,hsa-miR-96-5p,RAB27A,NaN,-0.496,0.004,Basal-like
4,hsa-miR-106b-5p,ATXN1,MIRT441140,-0.495,0.004,Basal-like
...,...,...,...,...,...,...
709,ITGA9,KEGG:05165,NaN,NaN,NaN,Luminal A
710,PPP2CB,KEGG:05165,NaN,NaN,NaN,Luminal A
711,HEY2,KEGG:05165,NaN,NaN,NaN,Luminal A
712,CDK6,KEGG:05165,NaN,NaN,NaN,Luminal A


In [11]:
concat_nodes_df

,id,label,type,qvalue,connected_component_id,is_in_largest_component,betweenness_centrality,closeness_centrality,degree_centrality,redundancy_coefficient,pathway_reach,functional_impact,group
0,hsa-miR-182-5p,hsa-miR-182-5p,MicroRNA,NaN,0.0,1.0,0.156133,0.259864,0.114094,0.044118,0.555556,0.365727,Basal-like
1,hsa-miR-18a-5p,hsa-miR-18a-5p,MicroRNA,NaN,0.0,1.0,0.107650,0.282127,0.040268,0.000000,0.666667,0.366496,Basal-like
2,hsa-miR-96-5p,hsa-miR-96-5p,MicroRNA,NaN,0.0,1.0,0.497522,0.375246,0.214765,0.030242,0.666667,0.402333,Basal-like
3,hsa-miR-106b-5p,hsa-miR-106b-5p,MicroRNA,NaN,0.0,1.0,0.491275,0.384306,0.268456,0.042308,0.777778,0.375430,Basal-like
4,hsa-miR-221-3p,hsa-miR-221-3p,MicroRNA,NaN,5.0,0.0,NaN,NaN,NaN,NaN,0.444444,0.468612,Basal-like
...,...,...,...,...,...,...,...,...,...,...,...,...,...
553,KEGG:04518,Integrin signaling,Pathway,0.018969,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Luminal A
554,KEGG:04820,Cytoskeleton in muscle cells,Pathway,0.018969,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Luminal A
555,KEGG:05202,Transcriptional misregulation in cancer,Pathway,0.018969,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Luminal A
556,KEGG:04151,PI3K-Akt signaling pathway,Pathway,0.024587,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Luminal A


In [12]:
node_prominence = compute_node_prominence(
    concat_edges_df,
    concat_nodes_df,
)

# Inspect programmatically
# specific_nodes = node_prominence["degree"]["classifications"]["specific"]

# print it
print_node_prominence(node_prominence)

===== Tripartite Networks - MicroRNA Nodes =====

##### Betweenness Centrality - Descending Ranking #####

> specific = 4
- hsa-miR-106a-5p: ['Basal-like']
- hsa-miR-182-5p: ['Basal-like']
- hsa-miR-29b-3p: ['Luminal A']
- hsa-miR-301a-3p: ['Luminal A']

> shared = 4
- hsa-miR-106b-5p: ['Basal-like', 'Luminal A']
- hsa-miR-141-3p: ['Basal-like', 'Luminal A']
- hsa-miR-200b-3p: ['Basal-like', 'Luminal A']
- hsa-miR-96-5p: ['Basal-like', 'Luminal A']

> shared_similarity = 4
- hsa-miR-106b-5p: ['Basal-like', 'Luminal A', 0.022]
- hsa-miR-141-3p: ['Basal-like', 'Luminal A', 0.045]
- hsa-miR-200b-3p: ['Basal-like', 'Luminal A', 0.208]
- hsa-miR-96-5p: ['Basal-like', 'Luminal A', 0.25]

##### Closeness Centrality - Descending Ranking #####

> specific = 5
- hsa-miR-106a-5p: ['Basal-like']
- hsa-miR-106b-5p: ['Basal-like']
- hsa-miR-200a-3p: ['Luminal A']
- hsa-miR-29b-3p: ['Luminal A']
- hsa-miR-429: ['Luminal A']

> shared = 3
- hsa-miR-200b-3p: ['Basal-like', 'Luminal A']
- hsa-miR-200c-3

In [13]:
node_prominence

{'MicroRNA': {'betweenness_centrality': {'descending': {'specific': {'hsa-miR-106a-5p': ['Basal-like'],
     'hsa-miR-182-5p': ['Basal-like'],
     'hsa-miR-29b-3p': ['Luminal A'],
     'hsa-miR-301a-3p': ['Luminal A']},
    'shared': {'hsa-miR-96-5p': ['Basal-like', 'Luminal A'],
     'hsa-miR-106b-5p': ['Basal-like', 'Luminal A'],
     'hsa-miR-200b-3p': ['Basal-like', 'Luminal A'],
     'hsa-miR-141-3p': ['Basal-like', 'Luminal A']},
    'shared_similarity': {'hsa-miR-96-5p': ['Basal-like', 'Luminal A', 0.25],
     'hsa-miR-106b-5p': ['Basal-like', 'Luminal A', 0.022],
     'hsa-miR-200b-3p': ['Basal-like', 'Luminal A', 0.208],
     'hsa-miR-141-3p': ['Basal-like', 'Luminal A', 0.045]}}},
  'closeness_centrality': {'descending': {'specific': {'hsa-miR-106b-5p': ['Basal-like'],
     'hsa-miR-106a-5p': ['Basal-like'],
     'hsa-miR-29b-3p': ['Luminal A'],
     'hsa-miR-429': ['Luminal A'],
     'hsa-miR-200a-3p': ['Luminal A']},
    'shared': {'hsa-miR-96-5p': ['Basal-like', 'Luminal 

In [14]:
nodes_with_npm_df = add_npm_results_to_nodes(
    concat_nodes_df,
    node_prominence,
)

save_group_network_data(concat_edges_df, nodes_with_npm_df)